# Module 23 — FastAPI

The same numbers as modules 21 and 22, served to a **program**. There is no HTML in
this module and no widget: `app.py` next to this notebook answers with JSON, and the
caller is code.

You can run it as a real server —

```console
uv run uvicorn 23_fastapi.app:app --reload
```

— and <http://127.0.0.1:8000/docs> is then a page nobody in this repository wrote.
Nothing below needs it. `TestClient` calls the application in this process, which is
Flask's `test_client()` from module 21 and Streamlit's `AppTest` from module 22, one
more time and for the same reason.

The reason this module exists is one sentence:

> **Your type hints stop being notation and become the program.**

In module 04 an annotation was documentation that mypy read, and deleting it changed
nothing about what ran. That is over.

In [ ]:
import sys
import warnings
from pathlib import Path

# starlette's own note about its own dependency. It goes to stderr, it changes nothing
# here, and it would otherwise appear above every cell below. Silenced by message, not
# by category, so a different deprecation would still get through -- and it has to come
# before the import that raises it, which is why the two imports below sit down here.
warnings.filterwarnings("ignore", message="Using `httpx`")

# The notebook runs in 23_fastapi/, so app.py is next to it. append, not insert: the
# installed course packages keep priority on the path.
sys.path.append(str(Path.cwd()))

from app import app  # noqa: E402
from fastapi.testclient import TestClient  # noqa: E402

client = TestClient(app)
print(client.get("/").json())

## The application, in one screen

`app.py` has six routes and about a hundred and fifty lines, half of them comment.
Everything it knows about sensors comes from `sensorreport` — the same package modules
21, 22, 24 and 25 use, so what differs between the five modules is only the
presentation.

The numbers are the ones from module 18's pandas and module 19's SQL.

In [ ]:
for path in ["/summary", "/current-faults"]:
    print(path)
    for entry in client.get(path).json():
        print("   ", entry)

## The annotation is the parser

Here is `fault_list` from `app.py`, with the comment removed:

```python
@app.get("/faults")
def fault_list(
    minimum: float = Query(default=LIMIT, ge=-50, le=200),
) -> list[ReadingOut]:
    above = [r for r in load_readings() if r.value is not None and r.value > minimum]
    ...
```

Everything on the wire is text. `?minimum=20` is the three characters `2`, `0` and
nothing else — there is no float in an HTTP request. Something has to convert it, and
in module 21 that something was you:

```python
minimum = float(request.args.get("minimum", "85"))   # Flask
```

Here it is the annotation. `minimum: float` is not a note about what the parameter
ought to be; it is the instruction that parses it. And `ge=-50, le=200` are not
documentation about the allowed range — they are the check.

One thing in that signature is not what it looks like. **`Query(...)` sits where the
default goes, and it is not one.** It is a metadata object FastAPI reads while it
builds the route and then removes; the actual default is the `default=` inside it. So
`minimum` is a `float` in the body, never a `Query` — and the parameter is still
optional, because `Query(default=...)` supplied one. It is there for what a bare
default cannot carry: the bounds, and the `description=` that ends up in the generated
document.

In [ ]:
print(client.get("/faults").status_code, len(client.get("/faults").json()))
print(
    client.get("/faults", params={"minimum": 20}).status_code,
    len(client.get("/faults", params={"minimum": 20}).json()),
)

In [ ]:
# `minimum` arrives as the four characters "warm". What status comes back?
assert client.get("/faults", params={"minimum": "warm"}).status_code == ...

## 422, and reading it

`422 Unprocessable Content` is the answer to a request that was **understood and is
not acceptable**. Not a 400 — the HTTP was fine, the query string parsed, the JSON was
valid JSON. The values were wrong.

The body is machine-readable on purpose. Every entry in `detail` says three things:

| field | what it is for |
| --- | --- |
| `loc` | where the bad value was: `["query", "minimum"]`, `["path", "index"]`, `["body", "celsius"]` |
| `type` | what kind of problem, as a stable string your code can branch on |
| `msg` | a sentence for a human |

A front end reads `loc` to decide which input field to highlight and `msg` to decide
what to write under it. That is why this is a list of objects and not a string: a
string would have been friendlier to read once and useless to build on.

In [ ]:
import json

for params in [{"minimum": "warm"}, {"minimum": 999}]:
    print(json.dumps(client.get("/faults", params=params).json(), indent=2))

### The refusal happened before your function

`fault_list` was never entered. FastAPI resolves and validates every parameter first,
and calls the function only if all of them came out.

That is the whole return on having a schema at the boundary: **inside the body,
`minimum` is a float and there is no other case.** No `try`, no `isinstance`, no
default-on-failure branch, and no error message of your own to write, translate, and
keep in step with whoever is calling you. The first line of the function can be the
work.

## It converted `"85.5"`, and it refused `"warm"`

Both are strings. So the rule is not "it refuses anything that is not already the
right type" — it converts, and the interesting question is where it stops. Measured:

In [ ]:
for raw in ["85.5", "1e2", "85.", "+85.5", " 85.5 ", "-20", "85.5 C", "85,5", "8.5.3", "0x50", ""]:
    response = client.get("/faults", params={"minimum": raw})
    if response.status_code == 200:
        print(f"  {raw!r:10} 200  {len(response.json())} readings")
    else:
        print(f"  {raw!r:10} 422  {response.json()['detail'][0]['type']}")

**The rule:** after stripping surrounding whitespace, the *whole* of the text has to be
a Python float literal. Two halves, and both matter.

- *The whole of it.* Not a prefix. `"85.5 C"` is refused, although a lenient parser
  could take the 85.5 and drop the ` C`. There is no partial success — which is also
  why `"8.5.3"` is refused rather than read as 8.5.
- *Float syntax, not locale.* `"85,5"` is refused. A German-speaking colleague reads
  that as eighty-five and a half; this parameter does not, because guessing which of
  `.` and `,` is the decimal separator would turn `85,5` into 85.5 or into 855
  depending on a setting nobody set.

So the criterion is **one unambiguous reading of the entire string**. Where two
readings exist, or where something would have to be discarded, it refuses.

That is exactly what five earlier tools in this course failed:

| module | the tool | what it does instead of failing |
| --- | --- | --- |
| 08 | `latin-1` | decodes any bytes; gives you `Â°C` |
| 16 | `requests` with no charset | falls back to Latin-1; the same `Â°C` |
| 17 | `html.parser` | repairs; four cells where there are two |
| 18 | `read_csv` | picks a type from the data; `.sum()` concatenates |
| 19 | SQLite | stores what it was given; `REAL` holds `'kaputt'` |

Every one of them had *a* reading for the input it was given. None of them had *one*.

### The bound is a second, separate check

`inf` and `nan` are valid float literals — `float("inf")` works — so the conversion
succeeds and something else has to stop them.

In [ ]:
for raw in ["inf", "nan"]:
    detail = client.get("/faults", params={"minimum": raw}).json()["detail"][0]
    print(f"  {raw!r:6} {detail['type']:18} {detail['msg']}")

`less_than_equal`, not `float_parsing`. Parsing worked; the bound refused.

Look at which value that catches. `nan` is the one module 18 warned about, where
`nan == nan` is false and every comparison against it is false — including
`nan <= 200`, which is why `le=200` rejects it. A route written as `minimum: float`
with no bound at all would have accepted `nan` and then reported zero faults out of
fifty, quietly, in a 200.

## The return annotation is the response schema

`-> SummaryOut` is not a note about what the function gives back. It is what FastAPI
validates the return value against, and what it puts in the generated document.

There is a `response_model=` argument for the cases the annotation cannot express, and
`app.py` never needs it: the annotation already said it.

Three versions of one unchanged function body follow. Predict the first.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel


class Out(BaseModel):
    tag: str
    value: float


demo = FastAPI()


@demo.get("/typed")
def typed() -> Out:
    return {"tag": "TH-04", "value": 93.5, "secret": "internal note"}


demo_client = TestClient(demo)

In [ ]:
# The function returned three keys, and `Out` declares two.
assert demo_client.get("/typed").json() == ...

In [ ]:
from typing import Any


@demo.get("/loose")
def loose() -> dict[str, Any]:
    return {"tag": "TH-04", "value": 93.5, "secret": "internal note"}


@demo.get("/bare")
def bare():
    return {"tag": "TH-04", "value": 93.5, "secret": "internal note"}


for path in ["/typed", "/loose", "/bare"]:
    print(f"  {path:8}", demo_client.get(path).json())

The same body, three annotations, and `secret` reaches the client in two of the three.

| return annotation | what came out | documented schema |
| --- | --- | --- |
| `-> Out` | `{'tag': 'TH-04', 'value': 93.5}` | `#/components/schemas/Out` |
| `-> dict[str, Any]` | all three keys | an object with any keys |
| none | all three keys | `{}` — nothing at all |

This is the change worth carrying out of the module. Until now an annotation could be
deleted or widened and the program behaved identically; only mypy noticed, and mypy is
advisory. Here **the annotation is the code that decides what leaves the process.**
`-> dict[str, Any]` is a data leak with a plausible cover story, in a one-line diff to
a signature.

And note which tool would have objected: none of them. `Any` is accepted everywhere by
construction — that is what it is for. So widening an annotation to make a type checker
stop complaining switches off the checker *and* the filter, at once.

## Bad input and bad output are not symmetric

A response that does not match its model is also caught. It is not a 422.

In [ ]:
@demo.get("/broken")
def broken() -> Out:
    return {"tag": "TH-04"}  # `value` is missing


# TestClient re-raises an exception from your function instead of turning it into a
# response -- in a test, the traceback is what you want.
try:
    demo_client.get("/broken")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).strip())

In [ ]:
# The same request through a client that does not re-raise: this is what a browser gets.
lenient = TestClient(demo, raise_server_exceptions=False)
response = lenient.get("/broken")
print(response.status_code, repr(response.text))

`500 Internal Server Error`, and not one word more. Compare the two:

| | who is wrong | what the client is told |
| --- | --- | --- |
| a bad **request** | the client | 422, with `loc`, `type` and `msg` naming the field |
| a bad **response** | you | 500, and nothing |

That asymmetry is correct, and it is worth being able to defend. The client has no
move here — it cannot supply your missing field. Naming which field of which internal
model was absent would tell a stranger the shape of your models and give them nothing
to act on. The traceback belongs in your log, and that is where it went.

**Detail in an error message is not politeness. It is a function of who can fix the
thing.**

## Two checkers reading one annotation

`-> Out` is read twice: by mypy before the program runs, and by Pydantic on every
request. They do not agree, and the disagreement is instructive. The cell below runs
mypy as a subprocess — module 20's `subprocess`, module 15's habit of shelling out to
a tool and reading what it says.

In [ ]:
import subprocess
import tempfile

CODE = """
from fastapi import FastAPI
from pydantic import BaseModel

class Out(BaseModel):
    tag: str
    value: float

app = FastAPI()

@app.get("/dict")
def as_dict() -> Out:
    return {"tag": "TH-04", "value": 93.5}       # a dict, not an Out

@app.get("/model")
def as_model() -> Out:
    return Out(tag="TH-04", value=93.5)          # the model
"""

with tempfile.TemporaryDirectory() as tmp:
    source = Path(tmp) / "routes.py"
    source.write_text(CODE)
    # --no-color-output, or the ANSI escapes end up in the cell output as noise.
    done = subprocess.run(
        [
            sys.executable,
            "-m",
            "mypy",
            "--strict",
            "--no-color-output",
            "--no-error-summary",
            str(source),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

print("exit", done.returncode)
for line in done.stdout.splitlines():
    print(" ", line.replace(str(source), "routes.py"))

One error, on `as_dict`, and nothing about `as_model`.

Both routes work at runtime — Pydantic takes the dict, validates it against `Out` and
builds the response. So this is not a bug mypy found; it is a disagreement about what
`-> Out` promises:

| | mypy, before running | Pydantic, per request |
| --- | --- | --- |
| a `dict` where `-> Out` is annotated | **error** | passes; validated and filtered |
| `minimum=warm` | cannot see it | **422**, field named |
| a response missing a field of the model | error, if the dict is a literal | **500** |
| `-> Any` | silent by construction | nothing left to check |

Which is right? Both, for their own question. mypy asks *does the code say what it
means* and a dict is not an `Out`. Pydantic asks *is this a valid response* and the
dict was. In this repository `mypy` does not check `23_fastapi`, so the choice is
yours — and returning the model, as `app.py` does, is the one that keeps both of them
useful.

## The document nobody wrote

`GET /openapi.json` describes the whole application: every path, every parameter,
every schema, every bound. There is no such file in this repository. It is built from
`app.py` when you ask for it, which is why it cannot fall behind the code.

`/docs` is a page that reads that document. So is the client generator in whatever
language your caller uses.

In [ ]:
spec = client.get("/openapi.json").json()

print("openapi", spec["openapi"])
print("title  ", spec["info"]["title"])
print("paths  ", sorted(spec["paths"]))
print("schemas", sorted(spec["components"]["schemas"]))

In [ ]:
# Where did gt=-50 and lt=200 on LimitIn.celsius end up?
print(json.dumps(spec["components"]["schemas"]["LimitIn"], indent=2))

`gt` and `lt` became `exclusiveMinimum` and `exclusiveMaximum` — JSON Schema's names
for the same two bounds. `ge` and `le` would have become `minimum` and `maximum`, which
is what happened to `Query(ge=-50, le=200)` on `/faults`. `required: ["celsius"]` is
there because `celsius` has no default and `note` does.

Nothing in that output was written by hand. It is the class, read.

## Where the document stops

Every route, and the statuses the document claims for it.

In [ ]:
for path, methods in sorted(spec["paths"].items()):
    for method, route in methods.items():
        print(f"  {path:22} {method:5} {sorted(route['responses'])}")

The three routes that take no parameters document a 200 and nothing else. The three
that take a parameter document a 422 as well — and nobody wrote that.

`/summary/{location}` is the interesting one. It has the 422, and `one_summary` in
`app.py` also raises a **404** — which is not there.

The reason is what FastAPI can see. It builds the document by **inspecting the
signature**: parameter names, annotations, defaults, return annotation. It never runs
the function and it does not read the body.

- The **422 is there** because it follows from the signature alone. A parameter that
  has to be parsed from text is a parameter that can fail to parse — which is exactly
  why `/`, `/summary` and `/current-faults` do not have one. Nobody wrote any of this;
  it is the signatures, read.
- The **404 is absent** because it is a `raise` inside the body, decided by data read
  at request time. Nothing in the signature implies it.

`responses={404: {"description": "no such location"}}` on the decorator puts it in.
But notice what is then true of it and was not true of the 422: **it can fall behind
the code.** The 422 is derived from the same annotation that produces the behaviour,
so if the annotation changes the document changes with it, in the same commit, with
nobody remembering. The hand-written 404 is a second statement of the same fact, and
two statements of one fact can disagree.

To a reader — and to a code generator — the generated half and the written half look
identical. That is the trap. The generated half is a specification; the written half is
a comment, and it needs what any comment needs: a test that fails when it goes stale.

`exercises/thinking.md` exercise 08 asks for two more ways a generated document can be
wrong. One of them is in this application, and it is about `SummaryOut.mean`.

## What FastAPI is not

It does not render pages. There are no templates here, and `/docs` is the only HTML in
the module — generated, not written. If your caller is a browser being used by a
person, this is the wrong tool, and modules 21 and 22 were the right ones.

| | Flask (21) | Streamlit (22) | FastAPI (23) |
| --- | --- | --- | --- |
| the caller | a browser | a browser | a program |
| what comes out | your HTML | Streamlit's page | JSON against a schema |
| bad input | your `if`, your message | there is no input to be bad | 422, field named, before your code |
| the annotations | ignored | ignored | are the parser and the filter |
| documentation | you write it | not applicable | generated, at `/docs` |
| this application | ~70 lines plus 4 templates | ~90 lines, no templates | ~150 lines, no templates |

**The heuristic: who is the caller?** A person needs a page, and the page is the work.
A program needs a contract, and the contract is the work — which is why this is the
module where the type hints became load-bearing. They *are* the contract, and there is
no second place where it is written down.

---

`exercises/` is next: seven files to fill in and two to think through. None of them
starts a server.

Module 24 leaves the web. A tkinter program does not answer requests; it **waits**, in
a loop it does not own, and that is the whole difficulty.